# Winning Prediction

In this notebook, we try to predict the outcome of a league of legends game through analyzing winsOver relationships paths in a knowledge graph. 

## Idea 

For every match we search for the pairs on the lanes (the two top-laners, the two supports, ...) 
and search the shortest path of (a:Player)-\[WINS_OVER...\]->(b:Player) between these players.   
The longer the path, the higher we expect this player to win against the other player.  

<br><div><img src="../assets/match_visualization.png" width="500"/></div>

Then, we check for every lane who we expect to be the winner and determine the team with more wins to win overall. 

As an example: In the following image,Team B wins the jungle, the bot-lane and the support. 
So its 3/5. Hence,  we expect team B to win. 

<br><div><img src="../assets/winner_looser_majority_voting.png" width="500"/></div>


In [1]:
from sqlalchemy import Engine
from tqdm import tqdm

from util.db.graph import Neo4JConnector
from util.db.postgres import PostgreSQLConnector

import pandas as pd

In [2]:
neo4j: Neo4JConnector = Neo4JConnector.create_from_config("../config.ini")
engine: Engine = PostgreSQLConnector.create_from_config("../config.ini").create_engine()

### Loading the Data

First, we load the matches and fetch the players position from the knowledge graph.

Then we transform the match in an easy format. 

In [3]:
# Load matches that are not covered in the knowledge graph 
df = pd.read_sql("SELECT * FROM challenger_matches WHERE NOT is_train", con=engine)
df

,cmid,id,game_creation,game_duration,game_id,game_mode,game_type,game_version,map_id,participant_identities,participants,platform_id,queue_id,season_id,status_message,status_status_code,teams,is_train
0,35330,35330,2020-03-28 20:03:52.933,00:20:43,4255914862,ARAM,MATCHED_GAME,10.6.314.4405,12,"[{'participantId': 1, 'player': {'platformId':...","[{'participantId': 1, 'teamId': 100, 'champion...",KR,450,13,0,0,"[{'teamId': 100, 'win': 'Fail', 'firstBlood': ...",False
1,37402,37402,2020-03-04 12:55:00.611,00:15:29,4194926592,ARAM,MATCHED_GAME,10.5.311.166,12,"[{'participantId': 1, 'player': {'platformId':...","[{'participantId': 1, 'teamId': 100, 'champion...",KR,450,13,0,0,"[{'teamId': 100, 'win': 'Fail', 'firstBlood': ...",False
2,44363,44363,2020-03-27 19:21:02.641,00:29:00,4252924880,ARAM,MATCHED_GAME,10.6.314.4405,12,"[{'participantId': 1, 'player': {'platformId':...","[{'participantId': 1, 'teamId': 100, 'champion...",KR,450,13,0,0,"[{'teamId': 100, 'win': 'Fail', 'firstBlood': ...",False
3,46838,46838,2020-03-22 10:50:23.676,00:03:20,4238153379,CLASSIC,MATCHED_GAME,10.6.313.8894,11,"[{'participantId': 1, 'player': {'platformId':...","[{'participantId': 1, 'teamId': 100, 'champion...",KR,430,13,0,0,"[{'teamId': 100, 'win': 'Fail', 'firstBlood': ...",False
4,47546,47546,2020-03-11 15:02:33.452,00:22:26,4211416607,ARAM,MATCHED_GAME,10.5.312.392,12,"[{'participantId': 1, 'player': {'platformId':...","[{'participantId': 1, 'teamId': 100, 'champion...",KR,450,13,0,0,"[{'teamId': 100, 'win': 'Win', 'firstBlood': T...",False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29174,145757,0,2020-01-29 15:29:24.569,00:21:45,4118590268,CLASSIC,MATCHED_GAME,10.2.305.4739,11,"[{'participantId': 1, 'player': {'platformId':...","[{'participantId': 1, 'teamId': 100, 'champion...",KR,420,13,0,0,"[{'teamId': 100, 'win': 'Win', 'firstBlood': T...",False
29175,145760,0,2020-01-18 02:28:36.651,00:28:57,4092375874,CLASSIC,MATCHED_GAME,10.1.303.9385,11,"[{'participantId': 1, 'player': {'platformId':...","[{'participantId': 1, 'teamId': 100, 'champion...",KR,420,13,0,0,"[{'teamId': 100, 'win': 'Win', 'firstBlood': F...",False
29176,145775,0,2020-02-17 05:04:03.240,00:16:10,4157911901,URF,MATCHED_GAME,10.3.307.7898,11,"[{'participantId': 1, 'player': {'platformId':...","[{'participantId': 1, 'teamId': 100, 'champion...",KR,900,13,0,0,"[{'teamId': 100, 'win': 'Fail', 'firstBlood': ...",False
29177,145788,0,2020-01-21 11:17:38.192,00:15:48,4100240254,CLASSIC,MATCHED_GAME,10.1.303.9385,11,"[{'participantId': 1, 'player': {'platformId':...","[{'participantId': 1, 'teamId': 100, 'champion...",KR,420,13,0,0,"[{'teamId': 100, 'win': 'Win', 'firstBlood': T...",False


In [4]:
# Load the players and receive their positions
query = """MATCH (p:Player {accountID: $account, platformID: $platform})-[hp:HAS_POSITION]->(pos:Position) RETURN pos.positionID"""

matches = {}
for _, row in tqdm(df.iterrows()):
    match = {"winning_team": {}, "loosing_team": {}}
    
    for participants, participant_identities in zip(row["participants"], row["participant_identities"]):
        win = participants["stats"]["win"]
        account, platform = participant_identities["player"]["accountId"], participant_identities["player"]["platformId"] 
        res = neo4j.exec(query, account=account, platform=platform)
        try:
            position = res.records[0]["pos.positionID"]
        except IndexError:
            continue
        
        match["winning_team" if win else "loosing_team"][position] = (account, platform)
    
    if len(match["winning_team"]) != 5 or len(match["loosing_team"]) != 5:
        continue
        
    matches[row['game_id']] = match

# visualize one match
matches[next(iter(matches))]

29179it [02:37, 185.38it/s]


{'winning_team': {'MID': ('JpBsef4xsPaWncKm7eZolLvrEDomP6sf6g0LHMM2yo_ESLY',
   'KR'),
  'JGL': ('TMslxDJdK_yPRE1IWbsDXG1ptC8cQMXpm7MK74H30UqcMsM', 'KR'),
  'SUP': ('tdPV9B3PVFP8G6gnAIJ2lCdZSXjgWgxOZFabXNxcdh00RVyPlipuiUAI', 'KR'),
  'BOT': ('e08jybn3DcAPjF0dMTWXjZCYd6TZSW4OrpQ5DNMaHp4vDKo', 'KR'),
  'TOP': ('nc-f7WhRepx47jP1BIGPk4cNwLcuDw6YH7zLbf7cQt0APGaGw-01znl7', 'KR')},
 'loosing_team': {'TOP': ('IIWG7FRiqE8CuZaCslKkNxkglwpgnePPxDslb2tv66FKrgo',
   'KR'),
  'SUP': ('KNmREBogecbDn1f-0rIP_QjQXB0evxeua6Aj-d1zyAT5-g6PhvWy7bDu', 'KR'),
  'BOT': ('DotaUIWsBXIuo0gdcYWikps5-nvtz6M0qq_MiLvQKU9n85U-Nojyr6z3', 'KR'),
  'MID': ('QrwhHNLZCGrRADqDGigOaOKnQRa81BEfwhn2D1mLl7FA9xA', 'KR'),
  'JGL': ('xSk5jPXNFKI3MJuNDFyX2-2dDrBVUAFkZ3DV_U6I5CuEi7c', 'KR')}}

### Predicting 

In this part, we search for a path between the players on the positions and perform the majority voting on the win. 

In [7]:
import numpy as np

def get_size_save(arr):
    return 0 if len(arr) == 0 else len(arr[0]["scores"])

# Query to extract the path from the knowledge graph
query = '''MATCH path = shortestPath(
            (p1:Player {accountID: $p1accountid, platformID: $p1platformid})
            -[w:WINS_OVER *]->
            (p2:Player {accountID: $p2accountid, platformID: $p2platformid}))
        RETURN [r in relationships(path) | r.rate] AS scores'''


correct_classified = []
for match in tqdm(matches):
    # Estimated probability, that the winning team actually wins 
    prop_win = 0
    
    for position in ["TOP", "MID", "JGL", "SUP", "BOT"]:
        p_winner = matches[match]["winning_team"][position]
        p_looser = matches[match]["loosing_team"][position]
        
        # get the shortest path for the winning player
        result_winner = neo4j.exec(query, p1accountid=p_winner[0], p1platformid=p_winner[1], p2accountid=p_looser[0], p2platformid=p_looser[1])
        # get the shortest path for the loosing player
        result_looser = neo4j.exec(query, p1accountid=p_looser[0], p1platformid=p_looser[1], p2accountid=p_winner[0], p2platformid=p_winner[1])
        
        if len(result_winner.records) != 0 or len(result_looser.records) != 0: 
            path_size_winner, path_size_looser = get_size_save(result_winner.records), get_size_save(result_looser.records)
            
            # if path of the winner is greater than the loosers path, add one to the winning probability
            if path_size_winner > path_size_looser:
                prop_win += 1
            # if path of the looser is greater than the winners path, subtract one from the winning probability
            elif path_size_winner < path_size_looser:
                prop_win += -1
    
    # No winner can be determined
    if prop_win == 0:
        continue
    
    # If more lanes of the winning team won -> predict a win
    correct_classified.append(prop_win > 0) 
    
number_classification = np.size(correct_classified)
correct_classification = np.sum(correct_classified)

f"Correctly Classified: {correct_classification} / {number_classification} ({round(correct_classification * 100 / number_classification, 2)}%)"

100%|██████████| 1035/1035 [00:13<00:00, 76.21it/s] 


'Correctly Classified: 354 / 726 (48.76%)'